In [1]:
import requests
import urllib.parse
from bs4 import BeautifulSoup
import mpl_finance
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from datetime import datetime
from openpyxl import load_workbook
import plotly.graph_objects as go
from plotly import subplots

%run _CrawBase.ipynb
%run _BaseInfo.ipynb

df_info=_baseInfo.copy()

C:\Users\user\anaconda3\lib\site-packages\mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


【craw_stock】 craw_stock stock_number!!!!!!!!! -->8487


In [2]:
def roc_to_gregorian(roc_date_str):
    # 解析字串，假設格式為 "ROC_YEAR/MM/DD"
    roc_year, month, day = roc_date_str.split('/')
    # 轉換為整數
    roc_year = int(roc_year)
    month = int(month)
    day = int(day)
    
    # ROC 年轉西元年
    gregorian_year = roc_year + 1911
    
    # 返回格式化的字串，或是你可以選擇返回 datetime.date 物件
    return pd.to_datetime(f"{gregorian_year:04d}-{month:02d}-{day:02d}")

In [3]:
def save_data_by_date(df, output_dir):
    # 清理无效字符（如 '*'）
    df['日期'] = df['日期'].astype(str).str.replace('*', '', regex=False)

    # 转换为日期格式
    df['日期'] = pd.to_datetime(df['日期'], format='%Y/%m/%d', errors='coerce')

    # 创建存储结果的文件夹
    os.makedirs(output_dir, exist_ok=True)

    # 按日期分组并保存为独立的 Excel 文件
    for date, group in df.groupby('日期'):
        print(date)
        print(roc_to_gregorian(min(end_time_list)))
        if(date>=roc_to_gregorian(min(end_time_list))):
            formatted_date = date.strftime('%Y-%m-%d')  # 格式化日期为文件名
            file_name = f"{output_dir}/{formatted_date}.xlsx"

            # 保存每个分组为单独的 Excel 文件
            group.to_excel(file_name, index=False)

In [4]:
# 計算 RSI 指標
def calculate_rsi(data, window=14):
    delta = data['收盤價'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [5]:
def get_interval(value_to_check, data):
    # 转换数据为浮点数并去除负值和零值
    cleaned_numbers = []
    for num in data:
        try:
            value = float(num)
            if value > 0:
                cleaned_numbers.append(value)
        except ValueError:
            # 忽略无法转换的字符串
            continue
    data= cleaned_numbers  
    
    value_to_check=float(value_to_check)
    
    # 计算均值和标准差
    mean = np.mean(data)
    std_dev = np.std(data, ddof=1)  # 使用 ddof=1 计算样本标准差

    confidence_levels = [0.1,0.30,0.50,0.65, 0.90, 0.99]
    
    best_interval = None
    for level in confidence_levels:
        z_score = stats.norm.ppf(1 - (1 - level) / 2)  # 获取 z 分数
        margin_of_error = z_score * std_dev
        
        lower_bound = mean - margin_of_error
        upper_bound = mean + margin_of_error
        # 检查 value_to_check 是否在当前置信区间内
        if lower_bound <= value_to_check <= upper_bound:
            best_interval = (level, lower_bound, upper_bound)
            break  # 如果找到包含 value_to_check 的置信区间，直接退出
    if best_interval:
        level, lower_bound, upper_bound = best_interval
        if mean <value_to_check:
            interval_type = "正區間"
        else:
            interval_type = "負區間"

        #print(f"数据点 {value_to_check} 落在置信水平 {level * 100}% 的{interval_type}: [{lower_bound:.2f}, {upper_bound:.2f}]")
        return (level,interval_type,lower_bound, upper_bound)
    else:
        if mean <value_to_check:
            interval_type = "正區間"
        else:
            interval_type = "負區間"
        
        return (1,interval_type,value_to_check, value_to_check)
        return None

In [6]:
# 清理函式，只對字串型態欄位進行處理
def clean_column(column):
    if column.dtype == 'object':
        return column.str.replace(',', '').replace('0', np.nan).replace('--', '').apply(pd.to_numeric, errors='coerce')
    return column  # 如果已經是數值型態，直接回傳

### Main 


In [7]:

def data_process(RowData_df_craw_stock):
    # 基本資料清理
    df = RowData_df_craw_stock.copy().drop_duplicates()
    df['日期'] = df['日期'].str.replace("＊", "", regex=False)
    df['日期'] = df['日期'].str.extract(r'(\d{2,3})/(\d{1,2}/\d{1,2})').apply(
        lambda x: f"{int(x[0]) + 1911}/{x[1]}", axis=1
    )
    df['年月日'] = df['日期']

    # 數值欄位轉換
    cols_to_clean = ['成交金額', '收盤價', '開盤價', '最低價', '最高價', '成交股數', '漲跌價差', '成交筆數']
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = clean_column(df[col])

    df['收盤價'] = pd.to_numeric(df['收盤價'], errors='coerce').fillna(method='ffill')
    
    # 前日收盤
    df['前日收盤價'] = df['收盤價'].shift(1)
    
    # ===== 技術指標計算區 =====
    ###########       【價】      ######################
    # MACD 計算　 MACD（指數平滑異同移動平均線）
    ema_12 = df['收盤價'].ewm(span=12, adjust=False).mean()
    ema_26 = df['收盤價'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['MACD-SL'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD-SL_max']= df['MACD-SL'].max()
    df['MACD-SL_min']= df['MACD-SL'].min()
    # ★ MACD 柱狀圖：快線 - 慢線
    df['MACD_hist'] = df['MACD'] - df['MACD-SL']
    df['MACD_hist_diff']=df['MACD_hist']-df['MACD_hist'].shift(1)
    
    
    # ========= MACD 口訣訊號 =========

    # 金叉／死叉（當天）
    macd_golden = (df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1))
    macd_death  = (df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))

    # 允許「近 K 天內有金叉 / 死叉」就算（而不是一定當天）
    K = 15
    macd_golden_recent = macd_golden.rolling(K, min_periods=1).max().astype(bool)
    macd_death_recent  = macd_death.rolling(K, min_periods=1).max().astype(bool)

    # =========== 抄底：前大後小 + 金叉 ===========

    # 柱狀在 0 軸下方，今天比昨天「沒那麼負」= 空方減弱
    hist_neg = df['MACD_hist'] < 0
    hist_contract = hist_neg & (df['MACD_hist'] > df['MACD_hist'].shift(1))

    df['MACD_bottom_signal'] = hist_contract & macd_golden_recent

    # =========== 逃頂：前高後低 + 死叉 + 放量 ===========

    # 柱狀在 0 軸上方，今天比昨天「更小」= 多頭動能衰退
    hist_pos = df['MACD_hist'] > 0
    hist_shrink = hist_pos & (df['MACD_hist'] < df['MACD_hist'].shift(1))

    # 放量：用近 10 日均量，之後你覺得太鬆再放大倍率
    vol_ma10 = df['成交金額'].rolling(window=10, min_periods=1).mean()
    vol_boost = df['成交金額'] > vol_ma10

    # 也允許「近幾天內有放量」，不要綁死當天
    vol_boost_recent = vol_boost.rolling(K, min_periods=1).max().astype(bool)

    df['MACD_top_signal'] = hist_shrink & macd_death_recent & vol_boost_recent

    # 如果想檢查有沒有抓到：
    #print("bottom_signal 數量：", df['MACD_bottom_signal'].sum())
    #print("top_signal 數量：", df['MACD_top_signal'].sum())
    # ========= =========
    
    
    
    
    # ========= 起漲區：MACD 站上 0 軸後一路走高 =========
    # 在 0 軸上方，且柱狀比前一天大 → 多頭動能在增強
    hist_pos = df['MACD_hist'] > 0
    hist_up  = df['MACD_hist'] > df['MACD_hist'].shift(1)

    # 整段「起漲區」：MACD 在 0 軸上方且動能在放大
    df['MACD_up_run'] = hist_pos & hist_up

    # 起漲「起點」：這段 run 的第一天
    up_run = df['MACD_up_run']
    df['MACD_rally_start'] = up_run & ~up_run.shift(1).fillna(False)
    
    
    # ========= =========
    
    
    
    # MACD 黃金交叉判斷
    df['MACD_golden_cross'] = ((df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1)))
    df['MACD_last_cross_date'] = df.loc[df['MACD_golden_cross'], '年月日'].max()

    try:
        MACD_last_cross_date=df.loc[df['MACD_golden_cross'], '年月日'].max()
        df['MACD_last_cross_date'] = MACD_last_cross_date
        df['MACD_last_cross_date收盤價']=df[df['年月日']==MACD_last_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_last_cross_date : {error}')   
    
    
    df['MACD_death_cross'] = ((df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1)))
    MACD_death_cross_date=df.loc[df['MACD_death_cross'], '年月日'].max()
    
    try:
        MACD_death_cross_date=df.loc[df['MACD_death_cross'], '年月日'].max()
        df['MACD_death_cross_d'] = MACD_death_cross_date
        df['MACD_death_cross_d收盤價']=df[df['年月日']==MACD_death_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_death_cross : {error}') 
        
        
    try:
        MACD_death_cross_date2=df.loc[df['MACD_death_cross'], '年月日'].iloc[-2]
        df['MACD_death_cross_d2'] = MACD_death_cross_date2
        df['MACD_death_cross_d2收盤價']=df[df['年月日']==MACD_death_cross_date2]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_death_cross_date2 : {error}')    

    # KD 計算
    low9 = df['收盤價'].rolling(window=9).min()
    high9 = df['收盤價'].rolling(window=9).max()
    df['%K'] = (df['收盤價'] - low9) / (high9 - low9) * 100
    df['%D'] = df['%K'].rolling(window=3).mean()

    # KD 黃金交叉判斷
    df['KD_golden_cross'] = ((df['%K'] > df['%D']) & (df['%K'].shift(1) <= df['%D'].shift(1)))
    
    # MA 計算　MA（移動平均線）與交叉
    df['MA_short'] = df['收盤價'].rolling(window=5).mean()
    df['MA_10'] = df['收盤價'].rolling(window=10).mean()
    df['MA_long'] = df['收盤價'].rolling(window=20).mean()
    df['MA_longlong'] = df['收盤價'].rolling(window=50).mean()
    df['MA_longlong_15'] = df['MA_longlong'].shift(15)
    
    df['MA_longlonglong'] = df['收盤價'].rolling(window=80).mean()
    df['MA_break'] = (df['MA_short'] > df['MA_long']) & (df['MA_short'].shift(3) <= df['MA_long'].shift(3))
    
    # MA 黃金交叉判斷
    df['MA_golden_cross'] = ((df['MA_long'] > df['MA_longlong']) & (df['MA_long'].shift(1) <= df['MA_longlong'].shift(1)))

    try:
        MA_last_cross_date=df.loc[df['MA_golden_cross'], '年月日'].max()
        df['MA_last_cross_date'] = MA_last_cross_date
        df['MA_last_cross_date收盤價']=df[df['年月日']==MA_last_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_last_cross_d2 : {error}')   
        
    try:
        MA_last_cross_date2=df.loc[df['MA_golden_cross'], '年月日'].iloc[-2]
        df['MA_last_cross_d2'] = MA_last_cross_date2
        df['MA_last_cross_d2收盤價']=df[df['年月日']==MA_last_cross_date2]['收盤價'].iloc[0]
    
    except Exception as error:
        print("")
        #print(f'Error get MA_last_cross_d2 : {error}')   
           
            
    # MA 死亡交叉判斷
    df['MA_death_cross'] = ((df['MA_long'] < df['MA_longlong']) & (df['MA_long'].shift(1) >= df['MA_longlong'].shift(1)))
    try:
        MA_death_cross_date=df.loc[df['MA_death_cross'], '年月日'].max()
        df['MA_death_cross_d'] = MA_death_cross_date
        df['MA_death_cross_d收盤價']=df[df['年月日']==MA_death_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_death_cross_date : {error}') 
        
    try:
        MA_death_cross_date2=df.loc[df['MA_death_cross'], '年月日'].iloc[-2]
        df['MA_death_cross_d2'] = MA_death_cross_date2
        df['MA_death_cross_d2收盤價']=df[df['年月日']==MA_death_cross_date2]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_death_cross_date2 : {error}')    
    
    
    # RSI
    df['RSI'] = calculate_rsi(df)
    df['RSI_rebound'] = (df['RSI'] > 30) & (df['RSI'].shift(1) <= 30)
    
    ###########       【量】      ######################
    
   # 價格變動方向加權成交金額（）１
    
    df['成交金額_5MA'] = df['成交金額'].rolling(window=5, min_periods=1).mean()

    # 如果成交金額是 NaN → 使用 5MA 補值
    df['Volume_Price_Change']= df['成交金額'].fillna(df['成交金額_5MA'])

    # 成交量移動平均與震盪指標
    df['Volume_MA_short'] = df['Volume_Price_Change'].rolling(window=5).mean()
    df['Volume_MA_long'] = df['Volume_Price_Change'].rolling(window=10).mean()
    df['Volume_MA_long50'] = df['Volume_Price_Change'].rolling(window=50).mean()
    df['Volume_Oscillator'] = (df['Volume_MA_short'] - df['Volume_MA_long']) 

    # VPC MACD 系列
    df['VPC_MACD'] = df['Volume_Price_Change'].ewm(span=9, adjust=False).mean() - \
                     df['Volume_Price_Change'].ewm(span=15, adjust=False).mean()
    df['VPC_SIGNAL'] = df['VPC_MACD'].ewm(span=9, adjust=False).mean()
    df['VPC_DIF'] = df['VPC_MACD'] - df['VPC_SIGNAL']
    
    '''
    df['VPC_MA_%']=(df['Volume_MA_short']-df['Volume_MA_long'])#/ abs(df['Volume_MA_long'])
    df['VPC_MA_ewmmean15']= df['VPC_MA_%'].fillna(0).ewm(span=9, adjust=False).mean()
    df['VPC_MA_rolling30']= df['VPC_MA_%'].fillna(0).rolling(window=30).mean()
    
    df['VPC_MA_check']=(df['VPC_MA_%']-df['VPC_MA_rolling30'])
    df['VPC_MA_check']=df['VPC_MA_check'].rolling(window=5).mean()
    df['VPC_MA_checkavg']=df['VPC_MA_check'].rolling(window=100).mean()
    '''
    

    
    #成交金額 均　
    ###### 計算最大與最小日期的相差天數
    df['年月日'] = pd.to_datetime(df['年月日'].astype(str).str.replace('*', '', regex=False), format='%Y-%m-%d')

    # 取最大與最小日期
    max_date = df['年月日'].max()
    min_date = df['年月日'].min()

    # 計算天數差
    diff_days = (max_date - min_date).days

    
    total_amount = df['成交金額'].sum()
    avg_amount = df['成交金額'].mean()
    avg_amount = df['成交金額'].mean()

    df['成交金額平均'] = avg_amount*1.5
    
        
    df['voc_cross'] = ((df['Volume_MA_short'] > df['成交金額平均']) & 
                       (df['Volume_MA_short'].shift(1) <= df['成交金額平均'].shift(1)))
    voc_cross_date=df.loc[df['voc_cross'], '年月日'].max()
    
    try:
        voc_cross_date=df.loc[df['voc_cross'], '年月日'].max()
        df['voc_cross_d'] = voc_cross_date
        #print(f' get voc_cross_d : {voc_cross_date}') 
    except Exception as error:
        print("")
        #print(f'Error get voc_cross_date : {error}') 
        
    
    
    ######

    # 交叉與門檻
   # 動能交叉幅度門檻：改小一些，讓條件不太嚴苛
    threshold = df['VPC_DIF'].rolling(window=10).std() * 0.1
    # volume_threshold 改用穩定的長期平均，避免波動太大
    df['volume_threshold'] = df['VPC_MACD'].ewm(span=12, adjust=False).mean()
    # 成交金額門檻：用絕對成交金額平均（非加權）來代表市場活躍程度
    volume_threshold = df['成交金額'].rolling(window=10).mean()
    
    # 黃金交叉觸發條件
    df['Volume_Price_Change_break'] = (
        (df['VPC_MACD'] > df['VPC_SIGNAL']) &
        (df['VPC_MACD'] > df['volume_threshold']) &
        (df['VPC_DIF'] > threshold) & 
        (df['成交金額'] > volume_threshold) & # 活躍才觸發
        (df['VPC_MACD'].shift(1) >= df['VPC_SIGNAL'].shift(1)) # 過去交叉持續
    )
    
    # 成交量擴增與變動率
    
    # 資料時間轉換與亮點處理
    df['年月日'] = pd.to_datetime(df['年月日'].astype(str).str.replace('*', '', regex=False), format='%Y-%m-%d')
    df, merged_intervals = get_highlight(df)
    ###########################################################################################
    
    # MACD 上升趨勢
    df['macd_golden_crosses_area'] = (
        (df['MACD'] > df['MACD-SL']) &
        (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))
    )

   
    ###########################################################################################
    # 判斷成交量震盪指標是否出現有效突破訊號
    df['VO_Positive'] = df['Volume_Oscillator'] > 5
    df['VO_Positive_Count'] = df['VO_Positive'].rolling(window=3).sum()
    '''
    # 同 buy_points
    df['buy_points'] =(
        (df['Volume_Oscillator'] > 5) &                              # 當日 VO 大於 5（濾除雜訊）
        (df['Volume_Oscillator'].shift(1) <= 0) &                   # 前一天小於等於 0（突破條件）
        (df['MA_short'] > df['MA_short'].shift(1)) &       # 均線上揚（確認價穩）
        (df['VO_Positive_Count'] >= 2)                              # 近三日至少兩日有效放量
    )
    '''
    # =====  UI 顯示顏色標記=====
    df['Bar_Color'] = df['收盤價'].diff().apply(lambda x: 'red' if x > 0 else 'green')
    
    # ===== 分類邏輯區 =====
    df['Full_Summary'] = ''
    df = full_technical_analysis(df)
    
    
    
    ###########################################################################################
    # 均線剛翻多（今天才第一次 MA_short > MA_long > MA_longlong）
    cond_ma_now = (df['MA_short'] > df['MA_long']) & (df['MA_long'] > df['MA_longlong'])
    ma_flip_today = cond_ma_now & ~(cond_ma_now.shift(1).fillna(False))

    # 量能放大（成交金額大於短期均量的 1.1 倍；原本 1.2 → 放寬一點）
    vol_expand_today = df['成交金額'] > (df['Volume_MA_short'] * 1.1)

    # MACD：改為「在零軸上方」即可（原本只抓『剛突破零軸』會漏掉後半段）
    macd_positive = df['MACD'] > 0
    # 若你仍想保留「剛破零軸」的嚴格條件，也一起算出來（供 rolling 使用）
    macd_zero_today = (df['MACD'] > 0) & (df['MACD'].shift(1) <= 0)

    # 允許近 K 天內先後滿足（K 可調）
    K = 3
    # 均線：用「正在多頭排列」以涵蓋後半段；若要嚴格起點，改回 ma_flip_today
    #ma_recent   = cond_ma_now.rolling(K, min_periods=1).max().astype(bool)
    ma_recent   = ma_flip_today.rolling(K, min_periods=1).max().astype(bool)
    vol_recent  = vol_expand_today.rolling(K, min_periods=1).max().astype(bool)
    # MACD：允許近 K 日內曾經『剛破零軸』或當下已 > 0
    macd_recent = ((macd_zero_today | macd_positive)
                   .rolling(K, min_periods=1).max().astype(bool))

    # 綜合條件（主升段起點/進場點）
    cond_entry = ma_recent & vol_recent & macd_recent
    df['cond_entry'] =( ma_recent & vol_recent & macd_recent )
    
    ###########################################################################################
    df['MA5_%']=(df['收盤價']-df['MA_short'])/ abs(df['MA_short'])
    df['MA5_%'] =( 100*df['MA5_%']).astype(float).apply(lambda x: f"{x:.2f}").astype(str)
    
    df['均價_%']=(df['MA_short']-df['MA_long'])/ abs(df['MA_long'])
    df['均價_%'] = (100*df['均價_%']).astype(float).apply(lambda x: f"{x:.2f}").astype(str)
    
    df['均價long_%']=(df['MA_long']-df['MA_longlong'])/ abs(df['MA_longlong'])
    df['均價long_%'] = (100*df['均價long_%']).astype(float).apply(lambda x: f"{x:.2f}").astype(str)
    
    df['MACD_%']=(df['MACD']-df['MACD-SL'])/ abs(df['MACD-SL'])
    df['MACD_%'] = (100*df['MACD_%']).astype(float).apply(lambda x: f"{x:.2f}").astype(str)
    
    
    # 先確保欄位是數值型態（目前你用 str(0)，會變成字串比較，這樣不對）
    df['MA5_%'] = df['MA5_%'].astype(float)
    df['均價_%'] = df['均價_%'].astype(float)
    df['均價long_%'] = df['均價long_%'].astype(float)
    df['MACD_%'] = df['MACD_%'].astype(float)

    # 各條件
    
    cond_a = (
        (df['MA5_%'] > 0) &
        (df['均價_%'] > 0) &
        (df['MACD'] > df['MACD-SL']) &
        (df['Volume_MA_short'] >df['Volume_MA_long50']*0.8 ) 
    )
    cond_b1 = (df['均價long_%'] > 0) & (df['MACD_%'] > 0) & (df['均價_%'] > 0)  
    cond_b2 = (df['均價long_%'] > 0) & (df['MACD_%'] > 0) & (df['均價_%'] < 0)  
    cond_c =  (df['均價long_%'] > 0) & (df['MACD_%'] < 0)
    cond_d1 = (df['均價long_%'] < 0) & (df['MACD_%'] > 0) & (df['均價_%'] > 0)  
    cond_d2 = (df['均價long_%'] < 0) & (df['MACD_%'] > 0) & (df['均價_%'] < 0)  
    cond_e =  (df['均價long_%'] < 0) & (df['MACD_%'] < 0)

    def add_tag(df, condition, label):
        # 僅在分類標籤中尚未包含該類別時才加入
        df.loc[condition & (~df['分類'].str.contains(label, na=False)), '分類'] += f'{label};'

    '''
    # 使用 np.select 依條件分類
    conditions = [cond_a,cond_b1,cond_b2, cond_c, cond_d1, cond_d2, cond_e]
    choices =    ['價_量', 'B1分流','B2分流', 'C分流', 'D1分流','D2分流', 'E分流']

    df['分類'] = np.select(conditions, choices, default='未分類')
    ''' 
    df['分類'] =''
    add_tag(df, cond_a, '價_量')
    add_tag(df, cond_b1, 'B1分流')
    add_tag(df, cond_b2, 'B2分流')
    add_tag(df, cond_c,  'C分流')
    add_tag(df, cond_d1, 'D1分流')
    add_tag(df, cond_d2, 'D2分流')
    add_tag(df, cond_e,  'E分流')

    ###########################################################################################

    return df

In [8]:
def full_technical_analysis(df):
    """
    統一處理流程：先進行 K棒分析，再執行市場分類與建議
    """
    df = analyze_candlestick(df)
    df = analyze_candlestick_multiday(df)
    df = apply_market_classification(df)
    return df

def analyze_candlestick(df):
    """
    根據 K 棒型態分析，標記常見型態並評估方向：
    - 長紅 / 長黑 K 棒
    - 十字線
    - 上影線長 / 下影線長
    - 多頭吞噬 / 空頭吞噬
    - 錘頭線 / 吊人線 / 流星線
    - 下影長 > 實體兩倍（新增）
    """
    df = df.copy()
    df['實體長度'] = abs(df['收盤價'] - df['開盤價'])
    df['K棒型態'] = ''

    # 基本形態
    df.loc[(df['收盤價'] > df['開盤價']) & (df['實體長度'] > (df['最高價'] - df['最低價']) * 0.7), 'K棒型態'] = '長紅K棒'
    df.loc[(df['收盤價'] < df['開盤價']) & (df['實體長度'] > (df['最高價'] - df['最低價']) * 0.7), 'K棒型態'] = '長黑K棒'
    df.loc[(df['實體長度'] <= (df['最高價'] - df['最低價']) * 0.1), 'K棒型態'] = '十字線'

    # 上下影線判斷
    df['上影'] = df['最高價'] - df[['收盤價', '開盤價']].max(axis=1)
    df['下影'] = df[['收盤價', '開盤價']].min(axis=1) - df['最低價']
    df.loc[df['上影'] > df['實體長度'], 'K棒型態'] += '|上影線長'
    df.loc[df['下影'] > df['實體長度'], 'K棒型態'] += '|下影線長'

    # 吞噬形態
    df['昨收'] = df['收盤價'].shift(1)
    df['昨開'] = df['開盤價'].shift(1)
    df['昨高'] = df[['昨收', '昨開']].max(axis=1)
    df['昨低'] = df[['昨收', '昨開']].min(axis=1)
    df['今高'] = df[['收盤價', '開盤價']].max(axis=1)
    df['今低'] = df[['收盤價', '開盤價']].min(axis=1)
    df.loc[(df['收盤價'] > df['開盤價']) & (df['昨收'] < df['昨開']) &
           (df['今高'] > df['昨高']) & (df['今低'] < df['昨低']), 'K棒型態'] += '|多頭吞噬'
    df.loc[(df['收盤價'] < df['開盤價']) & (df['昨收'] > df['昨開']) &
           (df['今高'] > df['昨高']) & (df['今低'] < df['昨低']), 'K棒型態'] += '|空頭吞噬'

    # 錘頭 / 吊人 / 流星
    df.loc[
        (df['實體長度'] < (df['最高價'] - df['最低價']) * 0.3) &
        (df['下影'] > df['實體長度'] * 2) &
        (df['上影'] < df['實體長度'] * 0.3), 'K棒型態'
    ] += '|錘頭線或吊人線'

    df.loc[
        (df['實體長度'] < (df['最高價'] - df['最低價']) * 0.3) &
        (df['上影'] > df['實體長度'] * 2) &
        (df['下影'] < df['實體長度'] * 0.3), 'K棒型態'
    ] += '|流星線'

    # K棒方向標註
    df['K棒方向'] = '中性'
    df.loc[df['K棒型態'].str.contains('長紅K棒|下影線長|多頭吞噬|錘頭線'), 'K棒方向'] = '正向'
    df.loc[df['K棒型態'].str.contains('長黑K棒|上影線長|空頭吞噬|流星線|吊人線'), 'K棒方向'] = '負向'
    df.loc[df['K棒型態'].str.contains('十字線'), 'K棒方向'] = '觀望'

    # ✅ 新增：下影線 > 實體長度 * 2
    df['K棒續強確認'] = ''
    df.loc[df['下影'] > df['實體長度'] * 2, 'K棒續強確認'] = '下影大於實體兩倍'

    return df

def analyze_candlestick_multiday(df):
    """
    判斷三日 K 棒型態（如晨星、暮星、三白兵、三隻烏鴉）與方向
    """
    df = df.copy()
    df['多日K棒型態'] = ''
    df['多日K棒方向'] = ''

    df['三白兵'] = (
        (df['收盤價'] > df['開盤價']) &
        (df['收盤價'].shift(1) > df['開盤價'].shift(1)) &
        (df['收盤價'].shift(2) > df['開盤價'].shift(2)) &
        (df['收盤價'] > df['收盤價'].shift(1)) &
        (df['收盤價'].shift(1) > df['收盤價'].shift(2))
    )
    df.loc[df['三白兵'], ['多日K棒型態', '多日K棒方向']] = ['|三白兵', '正向']

    df['三隻烏鴉'] = (
        (df['收盤價'] < df['開盤價']) &
        (df['收盤價'].shift(1) < df['開盤價'].shift(1)) &
        (df['收盤價'].shift(2) < df['開盤價'].shift(2)) &
        (df['收盤價'] < df['收盤價'].shift(1)) &
        (df['收盤價'].shift(1) < df['收盤價'].shift(2))
    )
    df.loc[df['三隻烏鴉'], ['多日K棒型態', '多日K棒方向']] = ['|三隻烏鴉', '負向']

    df['晨星'] = (
        (df['收盤價'].shift(2) < df['開盤價'].shift(2)) &
        (abs(df['收盤價'].shift(1) - df['開盤價'].shift(1)) < (df['最高價'].shift(1) - df['最低價'].shift(1)) * 0.1) &
        (df['收盤價'] > df['開盤價']) &
        (df['收盤價'] > (df['收盤價'].shift(2) + df['開盤價'].shift(2)) / 2)
    )
    df.loc[df['晨星'], ['多日K棒型態', '多日K棒方向']] = ['|晨星', '正向']

    df['暮星'] = (
        (df['收盤價'].shift(2) > df['開盤價'].shift(2)) &
        (abs(df['收盤價'].shift(1) - df['開盤價'].shift(1)) < (df['最高價'].shift(1) - df['最低價'].shift(1)) * 0.1) &
        (df['收盤價'] < df['開盤價']) &
        (df['收盤價'] < (df['收盤價'].shift(2) + df['開盤價'].shift(2)) / 2)
    )
    df.loc[df['暮星'], ['多日K棒型態', '多日K棒方向']] = ['|暮星', '負向']

    return df

def apply_market_classification(df):
    import pandas as pd

    df['Market_State'] = ''
    df.loc[(df['收盤價'] < df['開盤價']) & ((df['最高價'] - df['開盤價']) / df['開盤價'] > 0.03), 'Market_State'] += ',衝高回落'
    df.loc[(df['收盤價'] > df['前日收盤價']) & (df['RSI'] < 30), 'Market_State'] += ',低檔翻揚'
    df.loc[(df['收盤價'] > df['MA_long'] * 0.98) & (df['收盤價'] < df['MA_long'] * 1.02) & (df['RSI'].between(45, 55)), 'Market_State'] += ',盤整震盪'
    df.loc[(df['MA_short'] > df['MA_long']) & (df['MA_long'] > df['MA_longlong']), 'Market_State'] += ',多頭排列'
    df.loc[(df['MA_short'] < df['MA_long']) & (df['MA_long'] < df['MA_longlong']), 'Market_State'] += ',空頭排列'
    df.loc[(df['收盤價'] > df['前日收盤價']) & (df['成交金額'] > df['Volume_MA_long'] * 1.5), 'Market_State'] += ',爆量上攻'
    df['Market_State'] = df['Market_State'].str.lstrip(',')

    df['Buy_Signal'] = ''
    df.loc[
        (df['Market_State'].str.contains('多頭排列')) &
        (df['KD_golden_cross']) &
        (df['Market_State'].str.contains('爆量上攻')),
        'Buy_Signal'
    ] = '建議關注買點'

    df['Sell_Signal'] = ''
    df.loc[
        (df['Market_State'].str.contains('空頭排列')) &
        ((df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))) &
        (df['收盤價'] < df['MA_long']),
        'Sell_Signal'
    ] = '建議留意風險'
  
    df['Watch_Signal'] = ''    
    df.loc[
        (df['Market_State'].str.contains('盤整震盪')) &
        (~df['KD_golden_cross']) &
        (~df['RSI_rebound']) &
        (~((df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1)))),
        'Watch_Signal'
    ] = '觀望為宜'

    df['Reversal_Signal'] = ''
    df['RSI_diff'] = df['RSI'].diff()
    df.loc[
        (df['Market_State'].str.contains('空頭排列')) &
        (df['RSI'] < 30) &
        (df['RSI_diff'] > 5),
        'Reversal_Signal'
    ] = '可能出現反轉訊號'

    df['Action_Advice'] = ''
    df['Advice_Score'] = 0
    df.loc[df['Watch_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['觀望為宜', 1]
    df.loc[df['Buy_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['建議關注買點', 2]
    df.loc[df['Reversal_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['可能出現反轉訊號', 3]
    df.loc[df['Sell_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['建議留意風險', 4]

    df['Full_Summary'] = (
        '<b>分類：</b>' + df['Market_State'].fillna('') + '<br>' +
        '<b>建議：</b>' + df['Action_Advice'].fillna('') + '<br>' +
        '<b>評分：</b>' + df['Advice_Score'].astype(str) + '<br>' +
        '<b>K棒方向：</b>' + df.get('K棒方向', pd.Series([''] * len(df))) + '<br>' +
        '<b>多日K棒方向：</b>' + df.get('多日K棒方向', pd.Series([''] * len(df))) + '<br>' +
        '<b>K棒續強：</b>' + df.get('K棒續強確認', pd.Series([''] * len(df))) + '<br>' +
        '<b>多日K棒型態：</b>' + df.get('多日K棒型態', pd.Series([''] * len(df)))
    )
    
    df['GoDown'] = ''
    df.loc[
        (df['成交金額'] > df['Volume_MA_long']) &
        (df['收盤價'] < df['開盤價']),
        'GoDown'
    ] = '下跌帶量'
    
    
    is_range = df['Market_State'].str.contains('盤整震盪', na=False)
    df['盤整震盪_前15日次數'] = (is_range.rolling(window=15, min_periods=1).sum())
    
    is_range2 = df['Market_State'].str.contains('多頭排列', na=False)
    df['多頭排列_前5日次數'] = (is_range2.rolling(window=5, min_periods=1).sum())
    
    is_range3 = df['GoDown'].str.contains('下跌帶量', na=False)
    df['下跌帶量_前15日次數'] = (is_range3.rolling(window=15, min_periods=1).sum())

  
    return df


In [9]:
def save_plt_to_html(stock_number,stock_data,fig,text_area):
    try:
        directory = f"Html"    #---本地路徑
        _filename=GetStockInfoByID(stock_number).replace('*', '')
        #print(_filename)
        #檔案路徑設定
        _FilePath = f"{directory}/[{stock_number}]{_filename}.html"  #---本地路徑
        print(" Visit Google: http://localhost/IT//"+_FilePath)
        #確認資料夾是否已存在
        if not os.path.exists(directory):
            os.makedirs(directory)

        #++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++html處理額外新增
        #先存下html在取出，針對head 寫入
        # 需先存下html
        fig.write_html(_FilePath)

        # 读取现有的 HTML 文件
        with open(_FilePath, "r", encoding="utf-8") as file:
            soup = BeautifulSoup(file, "html.parser")

        # html insert 
        head = soup.find('head')
        head.append(BeautifulSoup(text_area, "html.parser"))
        #++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++完成html內容部分

        ## 保存修改后的 HTML 文件 所有的股票都會先存下
        with open(_FilePath, "w", encoding="utf-8") as file:
            file.write(str(soup))

        lastdata=stock_data.iloc[-1]
        
    except Exception as error:
        print(f'Error save_plt_to_html processing  {stock_number}: {error}')

In [10]:
# 找到所有 Upper -> Down 的時間段
def find_intervals(upper, down):
    intervals = []
    for u in upper:
        for d in down:     
            if d > u:
                intervals.append([u, d])
                break
    return intervals

# 合併重維時間段，同時處理重複的標記
def merge_intervals(intervals):
    if not intervals:
        return intervals
    
    # 按每個區間的开始日期排序
    intervals.sort(key=lambda x: x[0])
    merged = []

    for current in intervals:
        if not merged:
            merged.append(current)
        else:
            last = merged[-1]
            if current[0] <= last[1] and current[1] > last[1]:
                # 分割出重複部分，並新增一個新區間
                merged.append([current[0], last[1]])
                merged.append([last[1], current[1]])
            elif current[0] > last[1]:
                merged.append(current)
            else:
                # 更新最後一個區間的結束日期
                last[1] = max(last[1], current[1])

    return merged

In [11]:
def get_highlight(stock_data):
    try:

        stock_data['highlight']=False

        macd_death_crosses = stock_data[
                (stock_data['MACD'] < stock_data['MACD-SL']) &  # 當前 MACD 線低於信號線
                (stock_data['MACD'].shift(1) >= stock_data['MACD-SL'].shift(1))  # 前一日 MACD 線高於或等於信號線
            ]
        #
        key_area = stock_data[
            #(stock_data['MA_short'] > stock_data['MA_long']) &
            (stock_data['MA_short'] > stock_data['MA_10']) &
            #(stock_data['%K'] > stock_data['%D']) &
            #(stock_data['MACD-SL'] > 0) &
             (stock_data['MA_long'] > stock_data['MA_longlonglong']) &
            #(stock_data['MA_longlong'] > stock_data['MA_longlonglong']) &

            (stock_data['MACD_hist_diff'] > 0) &
            (stock_data['MACD-SL'] > 0) &
            (stock_data['MACD'] > stock_data['MACD-SL']) 
        ]
        #
        Upper= list(key_area['年月日'])
        Down =list(macd_death_crosses['年月日'])

        Upper.sort()
        Down.sort()

        Down.append(pd.Timestamp(datetime.today().date() + timedelta(days=1)))
        intervals = find_intervals(Upper, Down)
        merged_intervals = merge_intervals(intervals)

        temp_dates =stock_data['年月日'] 

        try:
            intervals_1 =merged_intervals[::-1][0:2][0]
            #前一個區間日
            stock_data['highlight_stardate_1'] =intervals_1[0]
            stock_data['highlight_enddate_1'] = intervals_1[1]
            stock_data['highlight_stardate收盤價_1']=stock_data[stock_data['年月日']==intervals_1[0]]['收盤價'].iloc[0]
            stock_data['highlight_enddate收盤價_1']=stock_data[stock_data['年月日']==intervals_1[1]]['收盤價'].iloc[0]

        except Exception as error:
            print("")
           # print(f'Error get_highlight  前n個區間日 processing : {error}')      
        
        try:
            intervals_2 =merged_intervals[::-1][0:2][1]
            #前二個區間日
            stock_data['highlight_stardate_2'] =intervals_2[0]
            stock_data['highlight_enddate_2'] = intervals_2[1]
            stock_data['highlight_stardate收盤價_2']=stock_data[stock_data['年月日']==intervals_2[0]]['收盤價'].iloc[0]
            stock_data['highlight_enddate收盤價_2']=stock_data[stock_data['年月日']==intervals_2[1]]['收盤價'].iloc[0]

        except Exception as error:
            print("")
            #print(f'Error get_highlight2  前n個區間日 processing : {error}')
        
        
        # 遍历 merged_intervals 进行筛选
        for start, end in merged_intervals:
            mask = (temp_dates >= start) & (temp_dates <= end)
            stock_data.loc[mask, 'highlight'] = True
            stock_data.loc[mask, 'highlight_date'] = start
            #stock_data.loc[mask, 'highlight_date'] = start 年月日
            #2025
            #print(start)
            #print(stock_data[stock_data['年月日']==start]['收盤價'].iloc[0])
            stock_data.loc[mask, 'highlight_收盤價'] = stock_data[stock_data['年月日']==start]['收盤價'].iloc[0]
            stock_data.loc[mask, 'highlight_enddate'] = end
        
        #print(key_area)
    except Exception as error:
            print(f'【要追】Error get_highlight processing : {error}')
        
       
    
    return stock_data,merged_intervals

In [12]:
def gen_html(stock_number,stock_data):
    N = 200  # 只畫最後 300 根，依需求調整
    stock_data = stock_data.tail(N).copy()
    
    # 計算 MACD 黃金交叉和死亡交叉
    # 黃金交叉：當前 MACD 線由下而上穿過信號線
    macd_golden_crosses = stock_data[
        (stock_data['MACD'] > stock_data['MACD-SL']) &  # 當前 MACD 線高於信號線
        (stock_data['MACD'].shift(1) <= stock_data['MACD-SL'].shift(1))  # 前一日 MACD 線低於或等於信號線
    ]
 
    # 死亡交叉：當前 MACD 線由上而下穿過信號線
    macd_death_crosses = stock_data[
        (stock_data['MACD'] < stock_data['MACD-SL']) &  # 當前 MACD 線低於信號線
        (stock_data['MACD'].shift(1) >= stock_data['MACD-SL'].shift(1))  # 前一日 MACD 線高於或等於信號線
    ]
    stock_data,merged_intervals=get_highlight(stock_data)
    
    cond_a=stock_data[
        (stock_data['Full_Summary'].str.contains('多頭排列', na=False)) &

        (stock_data['成交股數'].astype(float) > 100) &
       # (stock_data['level'].astype(float) > 0) &

        ((stock_data['Volume_MA_long50'].astype(float) /
        stock_data['成交金額平均'].replace(0, np.nan).astype(float)) > 0.5) &


        (stock_data['MACD_hist_diff'].astype(float) > 0) &
        (
            (stock_data['Volume_MA_short'].astype(float) > stock_data['Volume_MA_long50'].astype(float) * 0.9) &
            (stock_data['Volume_MA_long'].astype(float)  > stock_data['Volume_MA_long50'].astype(float) * 0.9)
        )
        ]

    
    

    '''
    # 判斷成交量震盪指標是否出現有效突破訊號
    buy_points = stock_data[
        (stock_data['Volume_Oscillator'] > 5) &                              # 當日 VO 大於 5（濾除雜訊）
        (stock_data['Volume_Oscillator'].shift(1) <= 0) &                   # 前一天小於等於 0（突破條件）
        (stock_data['MA_short'] > stock_data['MA_short'].shift(1)) &       # 均線上揚（確認價穩）
        (stock_data['VO_Positive_Count'] >= 2)                              # 近三日至少兩日有效放量
    ]
    '''
    
    ###############################################################################################
    ###############################################################################################
    # 創建圖表
    fig = subplots.make_subplots(rows=3, cols=1, 
                        subplot_titles=('股價與移動平均線'#, 'KD 指標'
                                        , 'MACD 指標(有價)','交易量(有量)', '成交金額與成交量震盪指標'),
                        shared_xaxes=True, 
                        vertical_spacing=0.1, 
                        specs=[[{"secondary_y": True}],[{"secondary_y": True}] ,
                               [{"secondary_y": True}]],
                        row_heights=[0.6, 0.2, 0.2]  # 调整各行的高度比例
                        )
    ###############################################################################################
    # 在圖上標註
    '''
    fig.add_trace(go.Scatter( x=stock_data[stock_data['cond_entry']]['年月日'],
                             y=stock_data[stock_data['cond_entry']]['收盤價'],
                            mode='markers',
                            marker_symbol='triangle-up',
                            marker_color='green',
                            marker_size=12,
                            name='主升段起點'  ), row=1, col=1)
    '''
    fig.add_trace(go.Scatter( x=stock_data[ (stock_data['Market_State'].str.contains('多頭排列'))]['年月日'],
                             y=stock_data[ (stock_data['Market_State'].str.contains('多頭排列'))]['收盤價'],
                            mode='markers',
                            marker_symbol='cross',
                            marker_color='#f0422b',
                            marker_size=10,
                            name='多頭排列'  ), row=1, col=1)
      
    
    #godown      GoDown 下跌帶量
    fig.add_trace(go.Scatter( x=stock_data[ (stock_data['GoDown'].str.contains('下跌帶量'))]['年月日'],
                             y=stock_data[ (stock_data['GoDown'].str.contains('下跌帶量'))]['收盤價'],
                            mode='markers',
                            marker_symbol='x',
                            marker_color='black',
                            marker_size=10,
                            name='下跌帶量'  ), row=1, col=1)
    fig.add_trace(go.Scatter( x=stock_data[ (stock_data['Market_State'].str.contains('盤整震盪'))]['年月日'],
                             y=stock_data[ (stock_data['Market_State'].str.contains('盤整震盪'))]['收盤價'],
                            mode='markers',
                            marker_symbol='cross',
                            marker_color='#ebbd67',
                            marker_size=8,
                            name='盤整震盪'  ), row=1, col=1)
    fig.add_trace(go.Scatter( x=cond_a['年月日'],
                             y=cond_a['收盤價'],
                            mode='markers',
                            marker_symbol='diamond',
                            marker_color='#f0422b',
                            marker_size=20,
                            name='cond_a'  ), row=1, col=1)
    
    
    #低檔翻揚  盤整震盪  
    
    
    
    # 添加股票箱型圖到第一圖
    fig.add_trace(go.Candlestick(x=stock_data['年月日'],
                             open=stock_data['開盤價'],
                             high=stock_data['最高價'],
                             low=stock_data['最低價'],
                             close=stock_data['收盤價'],
                             name='箱型圖',visible='legendonly',
                             increasing_line_color='red', 
                             decreasing_line_color='green',
                             increasing_fillcolor='rgba(255,0,0,0.3)',
                             decreasing_fillcolor='rgba(0,255,0,0.3)'), row=1, col=1)

    # 添加股價、短期和長期移動平均線、黃金交叉和死亡交叉到第一圖
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['收盤價'],                  
                             mode='lines', line_color='grey',
                             text=stock_data['Full_Summary'],name='股價'), row=1, col=1)# Trend
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MA_short'],                  
                             mode='lines', line_color='#ebbd67', 
                             name='MA 5 (MA_short)'), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MA_10'],                  
                             mode='lines', line_color='#de7e66', 
                             name='MA 10 (MA_short)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MA_long'],                  
                             mode='lines', line_color='blue', 
                             name='MA 20 (MA_long)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MA_longlong'],                  
                             mode='lines', line_color='red', 
                             name='MA 50 (MA_longlong)'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MA_longlonglong'],                  
                             mode='lines', 
                             line_color='pink',  
                             name='MA 80 (MA_longlonglong)'), row=1, col=1)
        
    
    fig.add_trace(go.Scatter(x=stock_data[stock_data['MA_break']]['年月日'],
                         y=stock_data[stock_data['MA_break']]['MA_long'],
                         mode='markers', marker_symbol="star", 
                         marker_color="red", marker_size=10,
                         name='MA_break',visible='legendonly'), row=1, col=1, secondary_y=False)
    
 
    # 添加成交金額、短期和長期移動平均線、成交量震盪指標以及買入點到第二圖
    fig.add_trace(go.Bar(x=stock_data['年月日'],
                         y=stock_data['成交金額'],
                         name='成交金額',
                         marker_color=stock_data['Bar_Color'],
                         opacity=0.5), row=1, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter( x=stock_data['年月日'], 
                              y=stock_data['成交金額平均'], 
                              name='成交金額平均',
                              mode='lines',line=dict(color='gold', width=3),  # 調整線條顏色與粗細
                              opacity=1),
                              row=1, col=1, secondary_y=True)
  
  
    i_row=1
    ###############################################################################################
    # 迴圈遍歷每個區間，將它們加入到圖形中
    for interval in merged_intervals:
        x0, x1 = interval
        fig.add_vrect(x0=x0, x1=x1,
                      annotation_text="--", annotation_position="top left",
                      fillcolor="#f6b26b", opacity=0.25, line_width=0, row=i_row, col=1)
    ###############################################################################################
    
    N = 200  # 只畫最後 300 根，依需求調整
    stock_view = stock_data.tail(N).copy()

    # ---- 顏色對應 ----
    color_map = {
        'B1分流': '#cc0000',  # 紅 #cc0000 
        'B2分流': '#f68b6b', 
        #'C分流' : '#3d85c6',  # 藍 #3d85c6
        'D1分流': '#fc8c14',  # 橘 #f6b26b
        'D2分流': '#c5db46',  # 橘 #c5db46
        
        #'E分流' : '#399e0d'   # 綠 #6aa84f
    }
 
    # ---- 將同一分流的連續天數「合併成區間」：每個分流最多產生 O(區段數) 個 vrect ----
    def build_spans(df, label_col='分類', date_col='年月日'):
        df = df.sort_values(date_col)
        # 只取有標籤的列
        s = df[label_col].where(df[label_col].notna())
        # 找出分段邊界：分流變化的位置
        boundary = (s != s.shift()).fillna(True)
        # 每段的起點 index
        starts = df.loc[boundary, date_col].values
        # 每段的終點：下一個起點的前一天（最後一段就是最後一天）
        ends = list(df.loc[boundary.shift(-1, fill_value=True), date_col].values)
        # 將標籤取出對應到每段
        labels = s.loc[boundary].values
        spans = []
        for lab, x0, x1 in zip(labels, starts, ends):
            if pd.isna(lab):  # 跳過未分類
                continue
            spans.append((lab, x0, x1))
        return spans

    spans = build_spans(stock_view, label_col='分類', date_col='年月日')
    
    
    max_close = stock_view['收盤價'].max()
    min_close = stock_view['收盤價'].min()

    strip = stock_view[['年月日', '分類']].copy()
    
    #strip_colors = [color_map.get(v, "#cccccc") for v in strip['分類']]
    strip_colors = [
        next((color for key, color in color_map.items() if key in v), "#cccccc")
        for v in strip['分類']
        ]

    # 設定頂端帶子的厚度：例如覆蓋最高價上方 1% 高度
    top0 = min_close * 0.99       # 帶子的底部（接近高點）
    top1 = max_close * 1.01       # 帶子的頂部
    strip['base'] = top0
    strip['y'] = top1 - top0      # Bar 的高度（細條）

    # y4 疊在主價軸上，使用自訂範圍（→ 不要 matches）
    fig.update_layout(
        yaxis4=dict(
            overlaying='y',
            anchor='x',
            side='right',
            range=[min_close*0.98, max_close*1.02],  # 自訂與主價軸相近
            showticklabels=False,
            showgrid=False
        ),
        barmode='overlay'  # 保險：避免與其他 bar 類 trace 相互堆疊
    )

    # 畫色帶（掛到 y4，指定 base 做頂端細條）
    strip_bar = go.Bar(
        x=strip['年月日'],
        y=strip['y'],              # 細條高度
        base=strip['base'],        # 關鍵：從 top0 開始畫
        marker=dict(color=strip_colors, line=dict(width=0)),
        opacity=0.25,
        name="分流色帶",
        hoverinfo='skip',
        showlegend=False,
        width=24*60*60*1000*0.9    # 日期軸下，條寬(毫秒) ≈ 0.9 個交易日
    )
    strip_bar.update(yaxis='y4')
    fig.add_trace(strip_bar, row=3, col=1)
       
    ###############################################################################################
    i_row=2
    ###################    
    #添加 MACD 指標、Signal Line、DIF 以及 MACD 的黃金交叉和死亡交叉到第四圖
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MACD'],
                             mode='lines', line_color='#ebbd67',
                             name='Diff(12,26)'), row=i_row, col=1)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MACD-SL'],
                             mode='lines', line_color='#67ceeb',
                             name='MACD(9)'), row=i_row, col=1)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MACD-SL_min'],
                             mode='lines', line_color='gray',
                             visible='legendonly',
                             name='MACD-SL_min'), row=i_row, col=1)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'],
                             y=stock_data['MACD-SL_max'],
                             mode='lines', line_color='gray',
                             name='MACD-SL_max'), row=i_row, col=1)
    
    
    # ★ MACD 柱狀圖：>0 用一色，<0 用另一色
    macd_hist = stock_data['MACD_hist']  # 就是 data_process 裡算好的那欄
    macd_colors = ['#f6b26b' if v >= 0 else '#9fc5e8' for v in macd_hist]

    fig.add_trace(go.Bar(x=stock_data['年月日'],
                         y=macd_hist,
                         marker_color=macd_colors,
                         name='MACD 柱狀',
                         opacity=0.6), row=i_row,col=1 )


    
    # ===== 口訣：抄底訊號（前大後小 + 金叉）=====
    bottom_pts = stock_data[stock_data['MACD_bottom_signal']]
    #print(bottom_pts)
    fig.add_trace( go.Scatter( x=bottom_pts['年月日'],
                                y=bottom_pts['MACD'],
                                mode='markers',
                                visible='legendonly',
                                marker_symbol='star',
                                marker_color='lime',
                                marker_size=14,
                                name='MACD 抄底訊號'),row=i_row,col=1 )

    # ===== 口訣：逃頂訊號（前高後低 + 死叉 + 放量）=====
    top_pts = stock_data[stock_data['MACD_top_signal']]
    #print(top_pts)
    fig.add_trace(go.Scatter(x=top_pts['年月日'],
                             y=top_pts['MACD'],
                                mode='markers',
                                visible='legendonly',
                                marker_symbol='star',
                                marker_color='magenta',
                                marker_size=14,
                                name='MACD 逃頂訊號'),row=i_row,col=1)
    #======================================================================
    
    
    # ===== 起漲區：用紅點標出整段 MACD 動能增加 =====
    up_pts = stock_data[stock_data['MACD_up_run']]
    fig.add_trace(
        go.Scatter(
            x=up_pts['年月日'],
            y=up_pts['MACD'],          # 或用 MACD_hist 都可以，看你喜歡
            mode='markers',
            visible='legendonly',
            marker_color='red',
            marker_size=8,
            name='MACD 起漲區'
        ),
        row=i_row,
        col=1
    )

    # ===== 起漲起點：用黃色記號標出第一根 =====
    start_pts = stock_data[stock_data['MACD_rally_start']]
    fig.add_trace(
        go.Scatter(
            x=start_pts['年月日'],
            y=start_pts['MACD'],
            mode='markers',
            visible='legendonly',
            marker_symbol='triangle-up',
            marker_color='yellow',
            marker_size=12,
            name='MACD 起漲起點'
        ),
        row=i_row,
        col=1
    )
    
    #======================================================================
    

    fig.add_trace(go.Scatter(x=macd_golden_crosses['年月日'], 
                             y=macd_golden_crosses['MACD'], 
                             mode='markers', marker_symbol="triangle-up", marker_color="red", marker_size=10,
                             name='MACD 黃金交叉'), row=i_row, col=1)
    fig.add_trace(go.Scatter(x=macd_death_crosses['年月日'], 
                             y=macd_death_crosses['MACD'], 
                             mode='markers', marker_symbol="triangle-down", marker_color="black", marker_size=10,
                             name='MACD 死亡交叉'), row=i_row, col=1)
    
    """
    fig.add_trace(go.Scatter(x=stock_data[stock_data['macd_golden_crosses_area']]['年月日'],
                             y=stock_data[stock_data['macd_golden_crosses_area']]['MACD'],
                             visible='legendonly', 
                             mode='markers', line_color='red',
                             name='Macd上升'), row=i_row, col=1)
  
    """
    fig.update_yaxes(title_text="MACD", row=i_row, col=1)
    
    ###############################################################################################
    i_row=3
    ###################    

    # 短交易量 Volume_MA_short
    # 遠交易量 Volume_MA_long

    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['成交金額'],
                             visible='legendonly', 
                             mode='lines', 
                             line=dict(width=1,  color='gray'),
                             name='成交金額'), row=i_row, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_short'],
                             mode='lines', 
                             line=dict(width=1, dash='dash', color='blue'),
                             name='_成交量MA5'),  row=i_row, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_long'],
                              mode='lines', 
                             line=dict(width=1, dash='dash', color='orange'),
                             name='_成交量MA10'), row=i_row, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_long50'],
                              mode='lines', 
                             text=stock_data['分類'],
                             line=dict(width=3,  color='red'),
                             name='_成交量MA50'), row=i_row, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter( x=stock_data['年月日'], 
                              y=stock_data['成交金額平均'], 
                              name='成交金額平均',
                              mode='lines',line=dict(color='gold', width=3),  # 調整線條顏色與粗細
                              opacity=1),
                              row=i_row, col=1, secondary_y=True)
    
    voc_cross = stock_data[stock_data['voc_cross']]
    fig.add_trace(
        go.Scatter(
            x=voc_cross['年月日'],
            y=voc_cross['Volume_MA_short'],
            mode='markers',
            marker_symbol='triangle-up',
            marker_color='yellow',
            marker_size=12,
            name='量起點'
        ),
        row=i_row,
        col=1, secondary_y=True
    )
    
   
    # 補上次 Y 軸標題（如需要）
    fig.update_yaxes(title_text="成交量指標", row=i_row, col=1, secondary_y=True)

   ###############################################################################################  
    
    # 設置 x 和 y 軸 label
    fig.update_yaxes(title_text="股價", row=1, col=1)
    fig.update_xaxes(title_text="日期", row=3, col=1)
    fig.update_xaxes(rangeslider_visible=False, row=1, col=1)
    for r in [1,2,3]:
        fig.update_xaxes(
            rangebreaks=[dict(bounds=["sat", "mon"])],
            row=r, col=1
        )

    ########################
    #  想在圖片右下角備註一些指標情況
    ########################
    
    # 定义灯号和文字的位置、颜色
    transparent_color = 'gray'
    indicator_labels = [
        'MACD 黃金交叉',
        'KD 黃金交叉',
        '觀察點-連續三日上升短期強勢信號',
        '轉強點-加權市場信號'
    ]
    indicator_colors = [
        'red' if stock_data['KD_golden_cross'].iloc[-1] else transparent_color,
        #'red' if stock_data['Signal_Balance_uptrend_3days'].iloc[-1] else transparent_color,
        #'red' if stock_data['Weighted_Signa_over_threshold'].iloc[-1] else transparent_color
    ]

    # 生成 HTML 内容
    html_content = ""
    for label, color in zip(indicator_labels, indicator_colors):
        html_content += f"""
        <div style="display: flex; align-items: center;">
            <div style="width: 10px; height: 10px; border-radius: 50%; background-color: {color}; margin-right: 8px;"></div>
            <span>{label}</span>
        </div><br>
    """
        
    html_iframe = f"""
    <div style="width: 100%; height: 250%; overflow: auto;">
        <div id="iframe-container" style="height:750px;display: none;">
            <iframe src="https://www.wantgoo.com/stock/{stock_number}" width="100%" height="100%" frameborder="0"></iframe>
        </div>
        <div id="iframe-container2" style="height:750px;display: none;">
            <iframe src="https://www.wantgoo.com/stock/{stock_number}/institutional-investors/trend" width="100%" height="100%" frameborder="0"></iframe>
        </div>
        <button onclick="toggleIframe()">顯示/隱藏 iframe</button>
        <button onclick="toggleIframe2()">法人動態</button>
        <a href="https://www.wantgoo.com/stock/{stock_number}"  target="_blank">玩股網</a>
        <a href=" https://tw.stock.yahoo.com/quote/{stock_number}.TW"  target="_blank">yahoo</a>
        <a href=" https://pscnetinvest.moneydj.com/z/zc/zca/zca.djhtm?a={stock_number}"  target="_blank">moneydj</a>
    </div>
    <script>
        function toggleIframe() {{
            var iframeContainer = document.getElementById('iframe-container');
            if (iframeContainer.style.display === 'none') {{
                iframeContainer.style.display = 'block';
            }} else {{
                iframeContainer.style.display = 'none';
            }}
        }}
           function toggleIframe2() {{
            var iframeContainer = document.getElementById('iframe-container2');
            if (iframeContainer.style.display === 'none') {{
                iframeContainer.style.display = 'block';
            }} else {{
                iframeContainer.style.display = 'none';
            }}
        }}
    </script>
    
    """

    # 添加 HTML 内容到图表
    html_text =html_iframe + '<br>'+ html_content+ '<br>'+ stock_data['Full_Summary'].iloc[-1]
    # MARK CONTENT 
    html_text =html_iframe +'<br>'
    # 创建新的文本区域并使用 CSS 定位到右下角
    text_area = f'''
    <style>
        .text-area {{
            position: fixed;
            bottom: 10px;
            right: 10px;
            background-color: white;
            padding: 10px;
            border: 1px solid black;
            box-shadow: 2px 2px 5px rgba(0,0,0,0.5);
            z-index: 1000;
            max-width: 300px;
            word-wrap: break-word;
        }}
    </style>
    <div class="text-area">
        {html_text}
    </div>
    '''

    save_plt_to_html(stock_number,stock_data,fig,text_area)

In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import threading

In [14]:
global df_save_df_to_excel
global df_save_old_RunRealTimeStock
df_save_df_to_excel=pd.DataFrame()
df_save_old_RunRealTimeStock=pd.DataFrame()

def save_to_df(_df):
    _df['年月日'] = _df['年月日'].dt.strftime('%Y-%m-%d')
    
    # 處理資料 df1
    df1=_df
    
    # 處理資料 df2
    _stock_data=_df
    to_RunRealTimeStock_excel=_df
    to_RunRealTimeStock_excel['now_price']=_stock_data['收盤價']
    to_RunRealTimeStock_excel['change_price']=_stock_data['漲跌價差']
    to_RunRealTimeStock_excel['change_quote'] = (_stock_data['漲跌價差'].astype(float, errors='ignore') / _stock_data['開盤價'].astype(float, errors='ignore')).replace([float('inf'), -float('inf'), float('nan')], 0) * 100

    to_RunRealTimeStock_excel['change_quote'] = to_RunRealTimeStock_excel['change_quote'].astype(float).apply(lambda x: f"{x:.2f}%")
    df2=to_RunRealTimeStock_excel[['日期','stock_number','now_price','change_price','change_quote']]

    # 更新全局 DataFrame
    global df_save_df_to_excel  
    if  df_save_df_to_excel.empty:
        df_save_df_to_excel=df1
    else :
        df_save_df_to_excel = pd.concat([df_save_df_to_excel, df1], ignore_index=True)
       
    global df_save_old_RunRealTimeStock  
    if  df_save_old_RunRealTimeStock.empty:
        df_save_old_RunRealTimeStock=df2
    else :
        df_save_old_RunRealTimeStock = pd.concat([df_save_old_RunRealTimeStock, df2], ignore_index=True)

In [15]:

# 定义保存 Excel 文件的函数
def save_process_data(stock_number, stock_data):
    try:
        if stock_data.empty:
            print(f'DataFrame is empty. Skipping save.{stock_number}')
            return
        
        level, interval_type, lower_bound, upper_bound = get_interval(stock_data.iloc[-1]['收盤價'], np.array(stock_data['收盤價']))
        stock_data['stock_number'] = stock_number
        stock_data['level'] = level
        stock_data['interval_type'] = interval_type
        stock_data['lower_bound'] = lower_bound
        stock_data['upper_bound'] = upper_bound
        
        #存excel 
        _max_end_date=max(end_time_list)
        _min_end_date=min(end_time_list) 
        max_end_date = f"{int(_max_end_date.split('/')[0]) + 1911}-{_max_end_date.split('/')[1]}-{_max_end_date.split('/')[2]}"
        min_end_date=f"{int(_min_end_date.split('/')[0]) + 1911}-{_min_end_date.split('/')[1]}-{_min_end_date.split('/')[2]}"

        stock_data = stock_data[
                    (stock_data['年月日'] <= max_end_date) & 
                    (stock_data['年月日'] >= min_end_date)
        ]
        save_to_df(stock_data)
        
    except Exception as error:
        print(f'Error in save_process_data for stock {stock_number}: {error}')

# 定义主要爬虫和处理函数
def process_stock_codes(stock_number):
    try:
        ## Test
        craw_stock_need_update=True
        #global _dummyData

        #craw_stock_need_update=False
        RowData_df_craw_stock, His_Stock, isSuccess = craw_stock(stock_number, start_month,(datetime.now() - timedelta(days=0)).strftime("%Y-%m-%d"),craw_stock_need_update)
        _dummyData=data_process(RowData_df_craw_stock)

        #存html 
        gen_html(stock_number, _dummyData)  
        save_process_data( stock_number, _dummyData)

    except Exception as error:
        print(f'Error processing stock {stock_number}: {error}')

# 分別處理不同的股票列表
def process_stock_list(stock_list):
    for stock_number in stock_list:
        process_stock_codes(stock_number)

In [16]:
## 【Run】 跑數據
start_month = '2025-03-01'

start_date = datetime.strptime(start_month, "%Y-%m-%d").date()
today = datetime.today().date()

def to_roc(date_obj):
    roc_year = date_obj.year - 1911
    return f"{roc_year}/{date_obj.month:02d}/{date_obj.day:02d}"

end_time_list = []
current_date = start_date
while current_date <= today:
    end_time_list.append(to_roc(current_date))
    current_date += timedelta(days=1)

end_time_list=end_time_list[-10:]

min(end_time_list) 


'115/02/12'

process_stock_codes('2330')




# Test
df_save_df_to_excel